# Data Preprocessing Sanity check
## Choose dataset, preprocessing method, and review results

This notebook provides a unified interface to:
1. Load data
2. Choose a preprocessing method (basic, **Llama**, or **Gemini API**)
3. **Test with a small sample first** (NUM_SAMPLES = 5)
4. Review and compare results
5. Run on full dataset when satisfied

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import pprint
import google.generativeai as genai
from dotenv import load_dotenv

# Add project root to path
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)  
sys.path.append(project_root)
sys.path.append(os.path.join(project_root, "src"))  # Add src to path

# Import the preprocessing module
from mosaic.preprocessing.preprocessing import (
    load_data,
    basic_preprocess,
    preprocess_with_local_llama,  # For local Llama model
    preprocess_with_gemini_api,   # For Google Gemini API
    compare_cleaning_results
)


print("Preprocessing module loaded successfully")

Preprocessing module loaded successfully


## Configuration

In [2]:
notebook_dir = Path.cwd()
project_root = notebook_dir.parent.parent
DATA_DIR = project_root / "DATA"
print(f"DATA_DIR set to: {DATA_DIR}")
if not DATA_DIR.exists():
    print("WARNING: DATA folder not found. Check your folder structure.")
else:
    #print the available raw datasets in the DATA_DIR/raw folder
    RAW_DIR = DATA_DIR / "raw"
    PREPROC_DIR = DATA_DIR / "preprocessed"
    available_datasets = [f.name for f in RAW_DIR.glob("*.csv")]
    print(f"Available raw datasets: {available_datasets}")

DATA_DIR set to: /Users/rb666/Projects/MOSAIC/DATA
Available raw datasets: ['dreamachine_DL_raw.csv', 'innerspeech_raw.csv', 'ganzfeld_GREEN_raw.csv', 'ganzfeld_RED_raw.csv', 'dreamachine_HS_raw.csv', 'innerspeech_raw_meta.csv', 'DMT_raw.csv', 'NDE_raw.csv', '5-MeO-DMT_raw.csv', 'MPE_raw.csv']


In [3]:
DATASETS = {}

if not DATA_DIR.exists():
    print("WARNING: DATA folder not found. Check your folder structure.")
elif not RAW_DIR.exists():
    print(f"WARNING: 'raw' folder not found inside DATA. ({RAW_DIR})")
else:
    # Find all CSVs that end with '_raw.csv'
    raw_files = list(RAW_DIR.glob("*_raw.csv"))
    
    print(f"Available raw datasets: {[f.name for f in raw_files]}")

    for file_path in raw_files:
        filename = file_path.name
        
        # Extract the name (remove '_raw.csv' from the end)
        dataset_name = filename.rsplit('_raw.csv', 1)[0]
        
        # Build the dictionary entry dynamically
        DATASETS[dataset_name] = {
            'input': filename,
            'output_api': f"{dataset_name}_cleaned_API.csv",
            'output_local': f"{dataset_name}_cleaned_llama.csv"
        }

    print("\n--- Generated Configuration ---")
    pprint.pprint(DATASETS)

Available raw datasets: ['dreamachine_DL_raw.csv', 'innerspeech_raw.csv', 'ganzfeld_GREEN_raw.csv', 'ganzfeld_RED_raw.csv', 'dreamachine_HS_raw.csv', 'DMT_raw.csv', 'NDE_raw.csv', '5-MeO-DMT_raw.csv', 'MPE_raw.csv']

--- Generated Configuration ---
{'5-MeO-DMT': {'input': '5-MeO-DMT_raw.csv',
               'output_api': '5-MeO-DMT_cleaned_API.csv',
               'output_local': '5-MeO-DMT_cleaned_llama.csv'},
 'DMT': {'input': 'DMT_raw.csv',
         'output_api': 'DMT_cleaned_API.csv',
         'output_local': 'DMT_cleaned_llama.csv'},
 'MPE': {'input': 'MPE_raw.csv',
         'output_api': 'MPE_cleaned_API.csv',
         'output_local': 'MPE_cleaned_llama.csv'},
 'NDE': {'input': 'NDE_raw.csv',
         'output_api': 'NDE_cleaned_API.csv',
         'output_local': 'NDE_cleaned_llama.csv'},
 'dreamachine_DL': {'input': 'dreamachine_DL_raw.csv',
                    'output_api': 'dreamachine_DL_cleaned_API.csv',
                    'output_local': 'dreamachine_DL_cleaned_llama.csv'},

## Step 1: Select Dataset and Preprocessing Method

### WORKFLOW FOR COMPARING MODELS:
1. Set `NUM_SAMPLES = 5` (or 10) to test
2. Run each method on the same sample
3. Compare results in Step 4
4. Choose best method
5. Set `NUM_SAMPLES = None` to run on full data

In [14]:
# ============================================
#  SELECT DATASET AND METHOD
# ============================================

# Choose dataset (see above abailable)
DATASET_CHOICE = 'MPE'

# Choose preprocessing method:
#   'local_llama'  → Local Llama 3 model (needs GPU: Metal/CUDA)
#   'gemini_api'   → Google Gemini API (needs GOOGLE_API_KEY)
METHOD_CHOICE = 'local_llama'

# FOR TESTING: Set to a number (5, 10, etc) or None for full dataset
# TIP: Test with 5-10 samples first to compare methods, then set to None
NUM_SAMPLES = 5

print(f"\n{'='*60}")
print(f"CONFIGURATION")
print(f"{'='*60}")
print(f"Dataset:        {DATASET_CHOICE}")
print(f"Method:         {METHOD_CHOICE}")
print(f"Samples:        {NUM_SAMPLES if NUM_SAMPLES else 'ALL (FULL DATASET)'}")
print(f"{'='*60}")

if NUM_SAMPLES:
    print(f"\nRunning in TEST mode with {NUM_SAMPLES} samples")
    print(f"  (Good for checking if everything works before full run)")
else:
    print(f"\nRunning on FULL DATASET")
    print(f"  (This may take a while)")


if METHOD_CHOICE == 'gemini_api':
    # Load .env file for API key
    load_dotenv()
    api_key = os.getenv("GOOGLE_API_KEY")
    
    if not api_key:
        print("Error: GOOGLE_API_KEY not found in .env file.")
        sys.exit(1)
    
    # Configure Gemini
    genai.configure(api_key=api_key)
    print("Google Gemini API configured successfully.")
    print(f"API Key found: {api_key[:5]}...*****")
    print("\n--- Available Gemini Models (generateContent) ---")
    
    # List Models
    try:
        found_any = False
        for m in genai.list_models():
            # Only show models that can generate text (chat models)
            if 'generateContent' in m.supported_generation_methods:
                print(f"  • {m.name}")
                found_any = True
        
        if not found_any:
            print("No models found. Check your API key permissions.")
            
    except Exception as e:
        print(f"Error listing models: {e}")


CONFIGURATION
Dataset:        MPE
Method:         local_llama
Samples:        5

Running in TEST mode with 5 samples
  (Good for checking if everything works before full run)


## Step 2: Load Data

In [15]:
# Build file paths
dataset_config = DATASETS[DATASET_CHOICE]
input_path = os.path.join(RAW_DIR, dataset_config['input'])

# Load data
df = load_data(input_path, text_column='reflection_answer', remove_na=True)

if df is not None:
    print(f"\nSuccessfully loaded {len(df)} records")
    print(f"DataFrame shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nFirst 5 rows (preview):")
    display(df.head(5))
else:
    print("Failed to load data. Check file path and try again.")

Loaded 830 rows from MPE_raw.csv

Successfully loaded 830 records
DataFrame shape: (830, 2)
Columns: ['id', 'reflection_answer']

First 5 rows (preview):


,id,reflection_answer
0,24,Pura consapevolezza è per me uno stato onniper...
1,26,"Die erste Erfahrung, auf die Ihre Beschreibung..."
2,31,"1. One night, before falling asleep, my mind b..."
3,32,Mi experiencia de conciencia ocurrió durante l...
4,33,The state of looking your own experience from ...


## Step 3: Run Preprocessing

In [16]:
if df is None:
    print("No data loaded. Run Step 2 first.")
else:
    df_to_process = df.head(NUM_SAMPLES).copy() if NUM_SAMPLES else df.copy()
    
    # ============================================
    # LOCAL LLAMA METHOD (using Llama 3)
    # ============================================
    if METHOD_CHOICE == 'local_llama':
        output_path = os.path.join(PREPROC_DIR, dataset_config['output_local'])
        if NUM_SAMPLES:
            output_path = output_path.replace('.csv', f'_{NUM_SAMPLES}_test.csv')
        
        df_to_process = preprocess_with_local_llama(
            csv_path=input_path,
            output_path=output_path,
            text_column='reflection_answer',
            num_samples=NUM_SAMPLES
        )
    
    # ============================================
    # GEMINI API METHOD
    # ============================================
    elif METHOD_CHOICE == 'gemini_api':
        print("Running GEMINI API preprocessing...")
        print(f"(Using Google Gemini with batch processing)\n")
        
        output_path = os.path.join(PREPROC_DIR, dataset_config['output_api'])
        if NUM_SAMPLES:
            output_path = output_path.replace('.csv', f'_{NUM_SAMPLES}_test.csv')
        
        df_to_process = preprocess_with_gemini_api(
            csv_path=input_path,
            output_path=output_path,
            text_column='reflection_answer',
            batch_size=20,
            num_samples=NUM_SAMPLES
        )
        print(f"\nGemini API preprocessing complete")
    
    else:
        print(f"Unknown method: {METHOD_CHOICE}")
        print(f"Choose from: 'local_llama', 'gemini_api'")


LOCAL LLAMA PREPROCESSING
Input file: MPE_raw.csv
Processing: 5 rows


llama_context: n_ctx_per_seq (4096) < n_ctx_train (8192) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_set_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_c4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f16                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64  


Starting text cleaning...


Cleaning texts with Llama: 100%|██████████| 5/5 [01:00<00:00, 12.01s/it]


Output saved to: /Users/rb666/Projects/MOSAIC/DATA/preprocessed/MPE_cleaned_llama_5_test.csv


## Step 4: Review Results

### Summary

In [7]:
if df_to_process is not None and 'cleaned_reflection' in df_to_process.columns:
    print(f"\n{'='*80}")
    print(f"RESULTS SUMMARY")
    print(f"{'='*80}")
    print(f"Total processed: {len(df_to_process)}")
    print(f"Columns: {df_to_process.columns.tolist()}")
    print(f"\nDataFrame Info:")
    print(df_to_process.info())


RESULTS SUMMARY
Total processed: 5
Columns: ['id', 'reflection_answer', 'cleaned_reflection']

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  5 non-null      int64 
 1   reflection_answer   5 non-null      object
 2   cleaned_reflection  5 non-null      object
dtypes: int64(1), object(2)
memory usage: 252.0+ bytes
None


### View Full Results Table

In [8]:
# Show all rows for review
print(f"\nAll {len(df_to_process)} processed records:")
display(df_to_process)


All 5 processed records:


,id,reflection_answer,cleaned_reflection
0,24,Pura consapevolezza è per me uno stato onniper...,Pure awareness for me is a pervasive state whe...
1,26,"Die erste Erfahrung, auf die Ihre Beschreibung...",The first experience to which your description...
2,31,"1. One night, before falling asleep, my mind b...","1. One night, before falling asleep, my mind b..."
3,32,Mi experiencia de conciencia ocurrió durante l...,My experience of consciousness occurred during...
4,33,The state of looking your own experience from ...,The state of looking at your own experience fr...


### Side-by-Side Comparison (Original vs Cleaned)

In [9]:
if 'reflection_answer' in df_to_process.columns and 'cleaned_reflection' in df_to_process.columns:
    print(f"\n{'='*80}")
    print("ORIGINAL vs CLEANED COMPARISON")
    print(f"{'='*80}\n")
    
    for i in range(min(5, len(df_to_process))):
        print(f"[Record {i+1}]")
        print(f"ORIGINAL:")
        original = str(df_to_process['reflection_answer'].iloc[i])
        print(f"  {original[:200]}..." if len(original) > 200 else f"  {original}")
        print(f"\nCLEANED:")
        cleaned = str(df_to_process['cleaned_reflection'].iloc[i])
        print(f"  {cleaned[:200]}..." if len(cleaned) > 200 else f"  {cleaned}")
        print(f"{'-'*80}\n")


ORIGINAL vs CLEANED COMPARISON

[Record 1]
ORIGINAL:
  Pura consapevolezza è per me uno stato onnipervasivo dove si fonde il confine tra me e la realtà intorno pur rimanendo chiari sia il mio corpo sia la realtà. Ha a che fare con un ritmo legato al respi...

CLEANED:
  Pure awareness for me is a pervasive state where the boundary between myself and the surrounding reality merges, while still keeping both my body and reality clear. It has to do with a rhythm linked t...
--------------------------------------------------------------------------------

[Record 2]
ORIGINAL:
  Die erste Erfahrung, auf die Ihre Beschreibung des "Bewussteins des Bewusstseins" zutreffend wuerde, hatte ich in noch sehr jungen Jahren. Bedingt durch den Tod meiner Mutter (als ich sieben Jahre war...

CLEANED:
  The first experience to which your description of 'consciousness of consciousness' would apply, I had at a very young age. Due to the death of my mother (when I was seven), I spent long afternoons and...

### Statistics

In [10]:
if 'reflection_answer' in df_to_process.columns and 'cleaned_reflection' in df_to_process.columns:
    print("\n" + "="*80)
    print("TEXT LENGTH STATISTICS")
    print("="*80 + "\n")
    
    original_lengths = df_to_process['reflection_answer'].astype(str).str.len()
    cleaned_lengths = df_to_process['cleaned_reflection'].astype(str).str.len()
    
    print(f"Original texts:")
    print(f"  Mean length: {original_lengths.mean():.0f} characters")
    print(f"  Min length: {original_lengths.min()} characters")
    print(f"  Max length: {original_lengths.max()} characters")
    
    print(f"\nCleaned texts:")
    print(f"  Mean length: {cleaned_lengths.mean():.0f} characters")
    print(f"  Min length: {cleaned_lengths.min()} characters")
    print(f"  Max length: {cleaned_lengths.max()} characters")
    
    print(f"\nDifference (cleaned - original):")
    diff = (cleaned_lengths - original_lengths)
    print(f"  Mean: {diff.mean():.0f} characters")
    print(f"  Min: {diff.min()} characters")
    print(f"  Max: {diff.max()} characters")


TEXT LENGTH STATISTICS

Original texts:
  Mean length: 1770 characters
  Min length: 118 characters
  Max length: 5449 characters

Cleaned texts:
  Mean length: 1711 characters
  Min length: 121 characters
  Max length: 5454 characters

Difference (cleaned - original):
  Mean: -58 characters
  Min: -320 characters
  Max: 19 characters


## Step 5: Run Full Dataset (when satisfied)

After testing with `NUM_SAMPLES = 5` and reviewing results above:

1. **If results look good**: Uncomment code below
2. **Change `NUM_SAMPLES = 5` to `NUM_SAMPLES = None`**
3. **Re-run from Step 1 through Step 3** to process full dataset

In [11]:


print("\nTo run the full dataset:")
print("  1. Go to Step 1 (Select Dataset and Method)")
print("  2. Change NUM_SAMPLES = 5 to NUM_SAMPLES = None")
print("  3. Run cells in Steps 1 through 3 again")
print("  4. Review final results in Step 4")


To run the full dataset:
  1. Go to Step 1 (Select Dataset and Method)
  2. Change NUM_SAMPLES = 5 to NUM_SAMPLES = None
  3. Run cells in Steps 1 through 3 again
  4. Review final results in Step 4


### Run basic (basic preprocessing and divide into sentences)

In [12]:
df_to_process['cleaned_reflection']

0    Pure awareness for me is a pervasive state whe...
1    The first experience to which your description...
2    1. One night, before falling asleep, my mind b...
3    My experience of consciousness occurred during...
4    The state of looking at your own experience fr...
Name: cleaned_reflection, dtype: object

In [13]:

print("Running BASIC preprocessing...")
texts = df_to_process['cleaned_reflection'].tolist()
df_to_process = basic_preprocess(texts, split_into_sentences=True, min_words=2)
print(f"Basic preprocessing complete")

#show the first 5 rows after basic preprocessing
print(f"\nFirst 5 rows after BASIC preprocessing:")
display(df_to_process.head(5))
    

Running BASIC preprocessing...

Successfully loaded 74 texts.
Threshold (min_words): 2
Removed short texts:   2 (2.7%)
Removed duplicates:    0
Final count:           72
Basic preprocessing complete

First 5 rows after BASIC preprocessing:


,sentences
0,Pure awareness for me is a pervasive state whe...
1,It has to do with a rhythm linked to the breat...
2,It is muffled and sharp at the same time.
3,The first experience to which your description...
4,Due to the death of my mother (when I was seve...
